# Triton Puzzles — Part 2: Medium-Hard (2D tiles, norms, matmul)

**Puzzles 13–21.** Move from 1D vectors to **2D tiles**: row-wise ops, softmax, RMSNorm/LayerNorm, outer products, transpose, and the canonical Triton matmul (naive → masked → fused).

Assumes you've done Part 1 (or are comfortable with `pid/offs/mask` and block reductions).


## 0. The GPU mental model (read this once)

Before Triton makes sense, you need a picture of the machine.

```
         GPU
 ┌──────────────────────────────────────────────┐
 │  SM 0   SM 1   SM 2  ...  SM N                │   <- ~80–140 SMs on modern GPUs
 │  ┌──┐  ┌──┐                                   │
 │  │  │  │  │   each SM runs many WARPS (32     │
 │  │  │  │  │   threads in lockstep, SIMT)       │
 │  └──┘  └──┘                                   │
 │   |     |                                     │
 │   shared mem / L1   (~100KB per SM, fast)     │
 └──┬──────┬─────────────────────────────────────┘
    │      │
    └──────┴──── L2 cache (tens of MB, shared)
               │
               └── HBM / global memory (slow, GB)
```

**Triton's bargain with you**: you don't write per-thread code (like CUDA), you write per-*program* code. A program ≈ a CUDA block ≈ a tile of work. Inside, you operate on *vectors* (`BLOCK_SIZE` elements). The compiler vectorizes across threads, picks register allocation, decides shared-memory staging, and does software pipelining.

Key Triton primitives you'll use everywhere:
- `pid = tl.program_id(axis=0)` — which tile am I?
- `offs = pid * BLOCK + tl.arange(0, BLOCK)` — the row of indices this tile owns.
- `mask = offs < N` — guard for the tail tile.
- `x = tl.load(ptr + offs, mask=mask, other=0.0)` — vectorized gather from HBM.
- `tl.store(ptr + offs, val, mask=mask)` — vectorized scatter.

🧠 **Fun fact**: `tl.arange(0, BLOCK)` must have a `BLOCK` that is a *power of two known at compile time*. That's because Triton lowers the vector to a fixed-shape MLIR tensor; the compiler needs the shape to pick layouts. The same is why `BLOCK_SIZE` is a `tl.constexpr`.

In [ ]:
import os, math, time
import torch
import triton
import triton.language as tl

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cpu':
    os.environ['TRITON_INTERPRET'] = '1'   # let kernels run on CPU for learning
    print('No GPU detected — using Triton interpreter mode. Slow but debuggable.')
else:
    print('Device:', torch.cuda.get_device_name(0))
    print('SMs   :', torch.cuda.get_device_properties(0).multi_processor_count)

torch.manual_seed(0)

def check(out, ref, atol=1e-3, rtol=1e-3, name=''):
    ok = torch.allclose(out, ref, atol=atol, rtol=rtol)
    diff = (out - ref).abs().max().item()
    print(f"{'✅' if ok else '❌'} {name}  max|Δ|={diff:.3e}")
    return ok


### A quick visualization of how `program_id` tiles a vector

```
 vector length N = 13,  BLOCK = 4   →  grid = ceil(13/4) = 4 programs

  index : 0  1  2  3 | 4  5  6  7 | 8  9 10 11 |12  X  X  X
  pid   :     0      |     1      |     2      |     3 (tail, masked)
```

Every program independently computes its slice. There is **no implicit communication between programs** — if you want a global reduction across tiles, you either do a second kernel or use atomics.

## Concept break: **2D programs and 2D tiles**

Most real kernels (norms, attention, matmul) operate on **rows of a matrix**. The pattern is:

- Grid `(num_rows,)`, one program per row.
- Inside each program: a 1D tile of size `BLOCK_N` covering the row.
- Row pointer: `x_ptr + row * stride_row + tl.arange(0, BLOCK_N) * stride_col`.

Tile layout for an `M × N` matrix, one program per row, BLOCK_N tiling the row:

```
        col → 0       BLOCK_N    2*BLOCK_N   ...
  row 0  [ tile(0,0)  tile(0,1)  tile(0,2)  ...]   ← pid=0 loops over col-tiles
  row 1  [ tile(1,0)  tile(1,1)  ...               ← pid=1
  row 2  ...
```

Sometimes one row fits in one tile (softmax over hidden_dim=4096 with BLOCK_N=4096); sometimes you loop. We'll do both.

## Puzzle 13 — Row sums of a 2D matrix (single tile per row)
Assume `N <= BLOCK_N` so each row fits in one tile. Output is `(M,)` with `out[i] = sum(X[i, :])`.

In [ ]:
@triton.jit
def k_row_sum(X_ptr,out_ptr,stride_m,M,N,BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N); mask = cols < N
    # TODO: row_ptr = X_ptr + row*stride_m + cols; x = tl.load(row_ptr, mask, other=0.0); store sum to out_ptr+row
    pass

def test_p13():
    M,N=128,1024; X=torch.randn(M,N,device=DEVICE); out=torch.empty(M,device=DEVICE)
    k_row_sum[(M,)](X,out,X.stride(0),M,N,BLOCK_N=1024)
    check(out, X.sum(1), atol=1e-2, name='P13 row_sum')
test_p13()


## Puzzle 14 — Single-row softmax (assume row fits in one tile)
Classic numerically-stable softmax: `x = x - max(x); e = exp(x); y = e / sum(e)`.

🧠 **Fun fact**: `torch.softmax` *also* does the subtract-max trick. Without it, `exp(1000)` overflows fp32. With it, the largest value becomes `exp(0)=1`, the rest are in `(0,1]`.

In [ ]:
@triton.jit
def k_softmax_row(X_ptr,Y_ptr,stride_m,M,N,BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N); mask = cols < N
    ptrs = X_ptr + row*stride_m + cols
    x = tl.load(ptrs, mask=mask, other=-float('inf'))
    # TODO: m = tl.max(x, 0); e = tl.exp(x - m); s = tl.sum(e, 0); y = e / s
    # TODO: store y at Y_ptr + row*stride_m + cols  (use the mask!)
    pass

def test_p14():
    M,N=64,512; X=torch.randn(M,N,device=DEVICE); Y=torch.empty_like(X)
    k_softmax_row[(M,)](X,Y,X.stride(0),M,N,BLOCK_N=512)
    check(Y, torch.softmax(X,1), atol=1e-5, name='P14 softmax_row')
test_p14()


## Puzzle 15 — RMSNorm
`y_i = x_i / sqrt(mean(x^2) + eps) * weight_i`. Single tile per row. Used in LLaMA / Qwen / DeepSeek.

In [ ]:
@triton.jit
def k_rmsnorm(X_ptr,W_ptr,Y_ptr,stride_m,N,eps,BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N); mask = cols < N
    xp = X_ptr + row*stride_m + cols
    x  = tl.load(xp, mask=mask, other=0.0)
    w  = tl.load(W_ptr + cols, mask=mask, other=0.0)
    # TODO: ms = tl.sum(x*x, 0) / N ; rstd = 1.0 / tl.sqrt(ms + eps); y = x * rstd * w
    # TODO: store y
    pass

def test_p15():
    M,N=32,4096; X=torch.randn(M,N,device=DEVICE); W=torch.randn(N,device=DEVICE); Y=torch.empty_like(X)
    k_rmsnorm[(M,)](X,W,Y,X.stride(0),N,1e-6,BLOCK_N=4096)
    ref = X * torch.rsqrt(X.pow(2).mean(-1,keepdim=True)+1e-6) * W
    check(Y, ref, atol=1e-4, name='P15 rmsnorm')
test_p15()


## Puzzle 16 — LayerNorm (mean and var)
`y = (x - mean) / sqrt(var + eps) * weight + bias`. Same tile pattern as RMSNorm, but two reductions: mean and second moment. Compute var as `mean(x^2) - mean(x)^2` to keep it one pass.

In [ ]:
@triton.jit
def k_layernorm(X_ptr,W_ptr,B_ptr,Y_ptr,stride_m,N,eps,BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N); mask = cols < N
    xp = X_ptr + row*stride_m + cols
    x  = tl.load(xp, mask=mask, other=0.0)
    # TODO: mean = tl.sum(x, 0)/N; var = tl.sum(x*x,0)/N - mean*mean
    # TODO: rstd = 1.0/tl.sqrt(var+eps); y = (x-mean)*rstd*tl.load(W_ptr+cols,mask,0.0) + tl.load(B_ptr+cols,mask,0.0)
    pass

def test_p16():
    M,N=16,2048; X=torch.randn(M,N,device=DEVICE); W=torch.randn(N,device=DEVICE); B=torch.randn(N,device=DEVICE)
    Y=torch.empty_like(X)
    k_layernorm[(M,)](X,W,B,Y,X.stride(0),N,1e-5,BLOCK_N=2048)
    ref = torch.nn.functional.layer_norm(X,(N,),W,B,1e-5)
    check(Y, ref, atol=1e-3, name='P16 layernorm')
test_p16()


## Puzzle 17 — Outer product
`Z[i,j] = x[i] * y[j]`. First real 2D tile. Trick: `x[:, None]` in Triton is `x[:, None]` — broadcasting works.

```
  x: (BM,)         y: (BN,)        x[:,None] * y[None,:]  →  (BM, BN)
  ┌─┐              ┌─────┐
  │a│              │1 2 3│   →   ┌──────┐
  │b│   ⊗          └─────┘       │ a 2a 3a│
  │c│                            │ b 2b 3b│
  └─┘                            │ c 2c 3c│
                                 └────────┘
```

In [ ]:
@triton.jit
def k_outer(x_ptr,y_ptr,Z_ptr,M,N,stride_zm,BM: tl.constexpr,BN: tl.constexpr):
    pm = tl.program_id(0); pn = tl.program_id(1)
    rm = pm*BM + tl.arange(0,BM); rn = pn*BN + tl.arange(0,BN)
    mm = rm < M; mn = rn < N
    x = tl.load(x_ptr+rm, mask=mm, other=0.0)        # (BM,)
    y = tl.load(y_ptr+rn, mask=mn, other=0.0)        # (BN,)
    # TODO: z = x[:, None] * y[None, :]   # (BM, BN)
    # TODO: store Z at Z_ptr + rm[:,None]*stride_zm + rn[None,:], mask = mm[:,None] & mn[None,:]
    pass

def test_p17():
    M,N=130,257; x=torch.randn(M,device=DEVICE); y=torch.randn(N,device=DEVICE)
    Z=torch.empty(M,N,device=DEVICE)
    k_outer[(triton.cdiv(M,32),triton.cdiv(N,64))](x,y,Z,M,N,Z.stride(0),BM=32,BN=64)
    check(Z, torch.outer(x,y), atol=1e-4, name='P17 outer')
test_p17()


## Puzzle 18 — 2D matrix transpose (tile-aware)
`B = A.T`. Each program owns a `BM × BN` tile of `A`, transposes in registers, writes into the symmetric `BN × BM` tile of `B`.

Why this is fun: **naive** transpose reads coalesced and writes strided (bad), or vice versa. Triton's compiler in many cases will rewrite the tile through shared memory automatically — but understanding the issue is the lesson.

In [ ]:
@triton.jit
def k_transpose(A_ptr,B_ptr,M,N,sa_m,sa_n,sb_n,sb_m,BM: tl.constexpr,BN: tl.constexpr):
    pm = tl.program_id(0); pn = tl.program_id(1)
    rm = pm*BM + tl.arange(0,BM); rn = pn*BN + tl.arange(0,BN)
    mm = rm<M; mn = rn<N
    a_ptrs = A_ptr + rm[:,None]*sa_m + rn[None,:]*sa_n
    a = tl.load(a_ptrs, mask=mm[:,None]&mn[None,:], other=0.0)  # (BM,BN)
    # TODO: write a's transpose into B at rows rn, cols rm
    #       b_ptrs = B_ptr + rn[:,None]*sb_n + rm[None,:]*sb_m  (shape BN,BM)
    #       tl.store(b_ptrs, tl.trans(a), mask = mn[:,None]&mm[None,:])
    pass

def test_p18():
    M,N=200,300; A=torch.randn(M,N,device=DEVICE); B=torch.empty(N,M,device=DEVICE)
    k_transpose[(triton.cdiv(M,32),triton.cdiv(N,32))](A,B,M,N,A.stride(0),A.stride(1),B.stride(0),B.stride(1),BM=32,BN=32)
    check(B, A.T.contiguous(), name='P18 transpose')
test_p18()


## Concept break: **2D programs and 2D tiles**

Most real kernels (norms, attention, matmul) operate on **rows of a matrix**. The pattern is:

- Grid `(num_rows,)`, one program per row.
- Inside each program: a 1D tile of size `BLOCK_N` covering the row.
- Row pointer: `x_ptr + row * stride_row + tl.arange(0, BLOCK_N) * stride_col`.

Tile layout for an `M × N` matrix, one program per row, BLOCK_N tiling the row:

```
        col → 0       BLOCK_N    2*BLOCK_N   ...
  row 0  [ tile(0,0)  tile(0,1)  tile(0,2)  ...]   ← pid=0 loops over col-tiles
  row 1  [ tile(1,0)  tile(1,1)  ...               ← pid=1
  row 2  ...
```

Sometimes one row fits in one tile (softmax over hidden_dim=4096 with BLOCK_N=4096); sometimes you loop. We'll do both.

## Puzzle 19 — Naive matmul (square, divisible sizes)
Assume `M,N,K` are divisible by `BM,BN,BK` so no masking. Get the structure right first.

In [ ]:
@triton.jit
def k_matmul_naive(A,B,C,M,N,K,sa_m,sa_k,sb_k,sb_n,sc_m,sc_n,
                   BM: tl.constexpr, BN: tl.constexpr, BK: tl.constexpr):
    pm = tl.program_id(0); pn = tl.program_id(1)
    rm = pm*BM + tl.arange(0,BM)
    rn = pn*BN + tl.arange(0,BN)
    rk = tl.arange(0,BK)
    a_ptrs = A + rm[:,None]*sa_m + rk[None,:]*sa_k
    b_ptrs = B + rk[:,None]*sb_k + rn[None,:]*sb_n
    acc = tl.zeros((BM,BN), dtype=tl.float32)
    for k in range(0, K, BK):
        # TODO: a = tl.load(a_ptrs); b = tl.load(b_ptrs); acc += tl.dot(a,b)
        # TODO: advance pointers along the K dimension
        pass
    c_ptrs = C + rm[:,None]*sc_m + rn[None,:]*sc_n
    tl.store(c_ptrs, acc)

def test_p19():
    M=N=K=256; A=torch.randn(M,K,device=DEVICE); B=torch.randn(K,N,device=DEVICE)
    C=torch.empty(M,N,device=DEVICE)
    k_matmul_naive[(M//64,N//64)](A,B,C,M,N,K,A.stride(0),A.stride(1),B.stride(0),B.stride(1),C.stride(0),C.stride(1),BM=64,BN=64,BK=32)
    check(C, A@B, atol=1e-2, rtol=1e-2, name='P19 matmul_naive')
test_p19()


## Puzzle 20 — Matmul with masking (arbitrary sizes)
Generalize P19 to any `M, N, K`. Mask `a` and `b` loads with `other=0.0` so out-of-bounds rows contribute zero to the dot. Mask `C` store on both axes.

In [ ]:
@triton.jit
def k_matmul_masked(A,B,C,M,N,K,sa_m,sa_k,sb_k,sb_n,sc_m,sc_n,
                    BM: tl.constexpr,BN: tl.constexpr,BK: tl.constexpr):
    pm = tl.program_id(0); pn = tl.program_id(1)
    rm = pm*BM + tl.arange(0,BM); rn = pn*BN + tl.arange(0,BN)
    acc = tl.zeros((BM,BN), dtype=tl.float32)
    for k0 in range(0, K, BK):
        rk = k0 + tl.arange(0,BK)
        # TODO: load A tile with mask (rm<M)[:,None] & (rk<K)[None,:]
        # TODO: load B tile with mask (rk<K)[:,None] & (rn<N)[None,:]
        # TODO: acc += tl.dot(a,b)
        pass
    c_ptrs = C + rm[:,None]*sc_m + rn[None,:]*sc_n
    tl.store(c_ptrs, acc, mask=(rm<M)[:,None] & (rn<N)[None,:])

def test_p20():
    M,N,K=257,193,131; A=torch.randn(M,K,device=DEVICE); B=torch.randn(K,N,device=DEVICE)
    C=torch.empty(M,N,device=DEVICE)
    k_matmul_masked[(triton.cdiv(M,64),triton.cdiv(N,64))](A,B,C,M,N,K,A.stride(0),A.stride(1),B.stride(0),B.stride(1),C.stride(0),C.stride(1),BM=64,BN=64,BK=32)
    check(C, A@B, atol=1e-2, rtol=1e-2, name='P20 matmul_masked')
test_p20()


## Puzzle 21 — Fused `Linear + bias + ReLU`
`Y = ReLU(X @ W + b)`. Take P20, broadcast `b` across rows, then `tl.maximum(acc, 0.0)` before the store. This is the kind of fusion that gives Triton its edge over `nn.Linear + relu` (one HBM round-trip on the output instead of two).

In [ ]:
@triton.jit
def k_linear_relu(X,W,bias,Y,M,N,K,sx_m,sx_k,sw_k,sw_n,sy_m,sy_n,
                  BM: tl.constexpr,BN: tl.constexpr,BK: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN)
    acc=tl.zeros((BM,BN),dtype=tl.float32)
    for k0 in range(0,K,BK):
        rk=k0+tl.arange(0,BK)
        a=tl.load(X+rm[:,None]*sx_m+rk[None,:]*sx_k, mask=(rm<M)[:,None]&(rk<K)[None,:], other=0.0)
        b=tl.load(W+rk[:,None]*sw_k+rn[None,:]*sw_n, mask=(rk<K)[:,None]&(rn<N)[None,:], other=0.0)
        acc += tl.dot(a,b)
    # TODO: load bias for cols rn (mask rn<N, other=0); add to acc broadcast; apply relu
    # TODO: store with full mask
    pass

def test_p21():
    M,N,K=128,256,512
    X=torch.randn(M,K,device=DEVICE); W=torch.randn(K,N,device=DEVICE); b=torch.randn(N,device=DEVICE)
    Y=torch.empty(M,N,device=DEVICE)
    k_linear_relu[(triton.cdiv(M,64),triton.cdiv(N,64))](X,W,b,Y,M,N,K,X.stride(0),X.stride(1),W.stride(0),W.stride(1),Y.stride(0),Y.stride(1),BM=64,BN=64,BK=32)
    check(Y, torch.relu(X@W+b), atol=1e-2, rtol=1e-2, name='P21 linear+bias+relu')
test_p21()


# 🔒 SOLUTIONS — stop scrolling unless you tried!


In [ ]:
# P13
@triton.jit
def sol_p13(X_ptr,out_ptr,stride_m,M,N,BLOCK_N: tl.constexpr):
    row = tl.program_id(0); cols=tl.arange(0,BLOCK_N); mask=cols<N
    x = tl.load(X_ptr+row*stride_m+cols, mask=mask, other=0.0)
    tl.store(out_ptr+row, tl.sum(x,0))

# P14
@triton.jit
def sol_p14(X_ptr,Y_ptr,stride_m,M,N,BLOCK_N: tl.constexpr):
    row = tl.program_id(0); cols=tl.arange(0,BLOCK_N); mask=cols<N
    ptrs = X_ptr+row*stride_m+cols
    x = tl.load(ptrs, mask=mask, other=-float('inf'))
    m = tl.max(x,0); e = tl.exp(x-m); s = tl.sum(e,0)
    tl.store(Y_ptr+row*stride_m+cols, e/s, mask=mask)

# P15
@triton.jit
def sol_p15(X_ptr,W_ptr,Y_ptr,stride_m,N,eps,BLOCK_N: tl.constexpr):
    row=tl.program_id(0); cols=tl.arange(0,BLOCK_N); mask=cols<N
    x=tl.load(X_ptr+row*stride_m+cols, mask=mask, other=0.0)
    w=tl.load(W_ptr+cols, mask=mask, other=0.0)
    ms = tl.sum(x*x,0)/N
    rstd = 1.0/tl.sqrt(ms+eps)
    tl.store(Y_ptr+row*stride_m+cols, x*rstd*w, mask=mask)

# P16
@triton.jit
def sol_p16(X_ptr,W_ptr,B_ptr,Y_ptr,stride_m,N,eps,BLOCK_N: tl.constexpr):
    row=tl.program_id(0); cols=tl.arange(0,BLOCK_N); mask=cols<N
    x=tl.load(X_ptr+row*stride_m+cols, mask=mask, other=0.0)
    mean = tl.sum(x,0)/N
    var  = tl.sum(x*x,0)/N - mean*mean
    rstd = 1.0/tl.sqrt(var+eps)
    w = tl.load(W_ptr+cols, mask=mask, other=0.0)
    b = tl.load(B_ptr+cols, mask=mask, other=0.0)
    tl.store(Y_ptr+row*stride_m+cols, (x-mean)*rstd*w + b, mask=mask)

# P17
@triton.jit
def sol_p17(x_ptr,y_ptr,Z_ptr,M,N,stride_zm,BM: tl.constexpr,BN: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN)
    mm=rm<M; mn=rn<N
    x=tl.load(x_ptr+rm, mask=mm, other=0.0)
    y=tl.load(y_ptr+rn, mask=mn, other=0.0)
    z = x[:,None]*y[None,:]
    tl.store(Z_ptr+rm[:,None]*stride_zm+rn[None,:], z, mask=mm[:,None]&mn[None,:])

# P18
@triton.jit
def sol_p18(A_ptr,B_ptr,M,N,sa_m,sa_n,sb_n,sb_m,BM: tl.constexpr,BN: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN)
    mm=rm<M; mn=rn<N
    a = tl.load(A_ptr+rm[:,None]*sa_m+rn[None,:]*sa_n, mask=mm[:,None]&mn[None,:], other=0.0)
    tl.store(B_ptr+rn[:,None]*sb_n+rm[None,:]*sb_m, tl.trans(a), mask=mn[:,None]&mm[None,:])
# P19
@triton.jit
def sol_p19(A,B,C,M,N,K,sa_m,sa_k,sb_k,sb_n,sc_m,sc_n,BM: tl.constexpr,BN: tl.constexpr,BK: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN); rk=tl.arange(0,BK)
    a_ptrs = A+rm[:,None]*sa_m+rk[None,:]*sa_k
    b_ptrs = B+rk[:,None]*sb_k+rn[None,:]*sb_n
    acc = tl.zeros((BM,BN),dtype=tl.float32)
    for k in range(0,K,BK):
        a = tl.load(a_ptrs); b = tl.load(b_ptrs)
        acc += tl.dot(a,b)
        a_ptrs += BK*sa_k
        b_ptrs += BK*sb_k
    tl.store(C+rm[:,None]*sc_m+rn[None,:]*sc_n, acc)

# P20
@triton.jit
def sol_p20(A,B,C,M,N,K,sa_m,sa_k,sb_k,sb_n,sc_m,sc_n,BM: tl.constexpr,BN: tl.constexpr,BK: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN)
    acc = tl.zeros((BM,BN),dtype=tl.float32)
    for k0 in range(0,K,BK):
        rk = k0+tl.arange(0,BK)
        a = tl.load(A+rm[:,None]*sa_m+rk[None,:]*sa_k, mask=(rm<M)[:,None]&(rk<K)[None,:], other=0.0)
        b = tl.load(B+rk[:,None]*sb_k+rn[None,:]*sb_n, mask=(rk<K)[:,None]&(rn<N)[None,:], other=0.0)
        acc += tl.dot(a,b)
    tl.store(C+rm[:,None]*sc_m+rn[None,:]*sc_n, acc, mask=(rm<M)[:,None]&(rn<N)[None,:])

# P21
@triton.jit
def sol_p21(X,W,bias,Y,M,N,K,sx_m,sx_k,sw_k,sw_n,sy_m,sy_n,BM: tl.constexpr,BN: tl.constexpr,BK: tl.constexpr):
    pm=tl.program_id(0); pn=tl.program_id(1)
    rm=pm*BM+tl.arange(0,BM); rn=pn*BN+tl.arange(0,BN)
    acc=tl.zeros((BM,BN),dtype=tl.float32)
    for k0 in range(0,K,BK):
        rk=k0+tl.arange(0,BK)
        a=tl.load(X+rm[:,None]*sx_m+rk[None,:]*sx_k, mask=(rm<M)[:,None]&(rk<K)[None,:], other=0.0)
        b=tl.load(W+rk[:,None]*sw_k+rn[None,:]*sw_n, mask=(rk<K)[:,None]&(rn<N)[None,:], other=0.0)
        acc += tl.dot(a,b)
    bvec = tl.load(bias+rn, mask=rn<N, other=0.0)
    acc = tl.maximum(acc + bvec[None,:], 0.0)
    tl.store(Y+rm[:,None]*sy_m+rn[None,:]*sy_n, acc, mask=(rm<M)[:,None]&(rn<N)[None,:])


print('Solutions for puzzles 13–21 defined.')


## Next up

Open **Part 3** (`triton_puzzles_3_hard.ipynb`) — conv1d, **online softmax**, **flash-attention-lite**, atomics, and the L2-swizzle trick. That's where it gets fun.
